# Module 12 — Agent Benchmarks & Enterprise Evals

> **SDKs:** `pydantic`, `dataclasses`, `statistics`

| Part | Topic |
|------|-------|
| **1** | Public Benchmarks — what SWE-bench really measures |
| **2** | Custom Enterprise Evals — golden datasets from prod traces |
| **3** | Trajectory Analysis — scoring the path, not just the outcome |


---
## Part 1 — Public Benchmarks: What They Measure (and Don't)

SWE-bench Verified tests an agent's ability to resolve GitHub Issues in open-source repos. It says nothing about whether the agent will obey your internal RBAC policies, call your proprietary APIs correctly, or stay within your rate limits.

In [1]:
from dataclasses import dataclass

@dataclass
class Benchmark:
    name: str
    what_it_tests: str
    what_it_misses: list[str]
    contamination_risk: str
    swe_bench_equiv: float   # 0-1

BENCHMARKS = [
    Benchmark(
        "SWE-bench Verified",
        "Resolve GitHub Issues in 300 curated open-source Python repos",
        ["Internal APIs", "RBAC enforcement", "Rate limits", "Production SLAs",
         "Multi-tenant isolation", "Proprietary data schemas"],
        "HIGH — training data cutoffs may include GitHub issues in the dataset",
        1.00,
    ),
    Benchmark(
        "WebArena",
        "Complete multi-step browser tasks (booking, form-filling)",
        ["API-only agents", "Multi-agent coordination", "SLA compliance",
         "Security policy adherence", "Audit trail generation"],
        "MEDIUM — specific websites' UI may appear in training data",
        0.72,
    ),
    Benchmark(
        "GAIA",
        "Complex multi-step QA with web search and tool use",
        ["Domain-specific tools", "Enterprise auth", "Real-time data",
         "Context window management at scale"],
        "LOW — synthetic tasks reduce contamination risk",
        0.48,
    ),
]

print("📋  Public Benchmark Analysis")
print("=" * 70)

for b in BENCHMARKS:
    print(f"\n  📊  {b.name}")
    print(f"  What it tests   : {b.what_it_tests}")
    print(f"  Contamination   : {b.contamination_risk}")
    print(f"  Missing coverage: {', '.join(b.what_it_misses[:3])} (and {len(b.what_it_misses)-3} more)")

print()
print("  ⚠️  DANGER: An agent that scores 50% on SWE-bench may score 0%")
print("  on your internal test suite, because internal tooling and policies")
print("  are never in the training data or public benchmarks.")


📋  Public Benchmark Analysis

  📊  SWE-bench Verified
  What it tests   : Resolve GitHub Issues in 300 curated open-source Python repos
  Contamination   : HIGH — training data cutoffs may include GitHub issues in the dataset
  Missing coverage: Internal APIs, RBAC enforcement, Rate limits (and 3 more)

  📊  WebArena
  What it tests   : Complete multi-step browser tasks (booking, form-filling)
  Contamination   : MEDIUM — specific websites' UI may appear in training data
  Missing coverage: API-only agents, Multi-agent coordination, SLA compliance (and 2 more)

  📊  GAIA
  What it tests   : Complex multi-step QA with web search and tool use
  Contamination   : LOW — synthetic tasks reduce contamination risk
  Missing coverage: Domain-specific tools, Enterprise auth, Real-time data (and 1 more)

  ⚠️  DANGER: An agent that scores 50% on SWE-bench may score 0%
  on your internal test suite, because internal tooling and policies
  are never in the training data or public benchmarks.


---
## Part 2 — Custom Enterprise Evals: Production Trace Golden Dataset

The highest quality evaluation set comes from your own production traces. Anonymize them, attach expected outcomes, and run them in CI/CD on every model upgrade.

In [2]:
import json, hashlib
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ProductionTrace:
    trace_id: str
    input: dict             # raw agent input
    tool_calls: list[dict]  # recorded tool call sequence
    output: dict            # actual agent output
    human_verdict: str      # "correct" | "incorrect" | "needs_review"
    tenant_id: str

@dataclass
class AnonymizedEvalCase:
    eval_id: str
    input: dict             # PII removed
    expected_tool_sequence: list[str]
    expected_output_schema: dict   # must match this JSON schema
    human_verdict: str

def anonymize_trace(trace: ProductionTrace) -> AnonymizedEvalCase:
    """
    Remove PII and internal identifiers from production traces.
    Anonymize tenant IDs with one-way hash.
    """
    anon_tenant = "tenant-" + hashlib.sha256(trace.tenant_id.encode()).hexdigest()[:8]
    
    clean_input = {
        k: v for k, v in trace.input.items()
        if k not in ["user_email", "user_name", "ip_address", "credit_card"]
    }
    clean_input["tenant_id"] = anon_tenant
    
    return AnonymizedEvalCase(
        eval_id="eval-" + trace.trace_id[:8],
        input=clean_input,
        expected_tool_sequence=[t["name"] for t in trace.tool_calls],
        expected_output_schema={
            "type": "object",
            "required": list(trace.output.keys()),
        },
        human_verdict=trace.human_verdict,
    )

# ─── Build golden dataset ─────────────────────────────────────────────────────
raw_traces = [
    ProductionTrace("trc-001", 
        input={"service": "checkout-ui", "user_email": "jane@acme.com", "tenant_id": "northstar-eu-001"},
        tool_calls=[{"name": "query_metrics"}, {"name": "get_deployment"}, {"name": "propose_revert"}],
        output={"hypothesis": "v2.1 broke 3DS", "confidence": "HIGH", "proposal": "revert"},
        human_verdict="correct", tenant_id="northstar-eu-001"),
    ProductionTrace("trc-002",
        input={"service": "billing-api", "user_email": "bob@globex.com", "tenant_id": "globex-002"},
        tool_calls=[{"name": "query_metrics"}, {"name": "search_logs"}],
        output={"hypothesis": "UNKNOWN — insufficient data", "confidence": "LOW", "proposal": "escalate"},
        human_verdict="correct", tenant_id="globex-002"),
]

golden_dataset = [anonymize_trace(t) for t in raw_traces]

print("🏆  Enterprise Eval Golden Dataset")
print("=" * 60)
print(f"  Raw traces   : {len(raw_traces)}")
print(f"  Eval cases   : {len(golden_dataset)}")
print()

for case in golden_dataset:
    print(f"  [{case.eval_id}]")
    print(f"    Input                  : {case.input}")
    print(f"    Expected tools         : {case.expected_tool_sequence}")
    print(f"    Human verdict          : {case.human_verdict}")
    print()

print("  ✅  Golden dataset ready for CI/CD attachment.")
print("  Run against every new model version or agent code change.")


🏆  Enterprise Eval Golden Dataset
  Raw traces   : 2
  Eval cases   : 2

  [eval-trc-001]
    Input                  : {'service': 'checkout-ui', 'tenant_id': 'tenant-8f81f3f7'}
    Expected tools         : ['query_metrics', 'get_deployment', 'propose_revert']
    Human verdict          : correct

  [eval-trc-002]
    Input                  : {'service': 'billing-api', 'tenant_id': 'tenant-5d0572f8'}
    Expected tools         : ['query_metrics', 'search_logs']
    Human verdict          : correct

  ✅  Golden dataset ready for CI/CD attachment.
  Run against every new model version or agent code change.


---
## Part 3 — Trajectory Analysis: Scoring the Path

Binary outcome evaluation (Did it succeed?) misses critical quality signals. An agent that achieved the right outcome by getting lucky on a hallucinated tool call is a production liability.

In [3]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class TrajectoryStep:
    step: int
    action: str    # "think" | "tool_call" | "output"
    content: str
    grounded: bool  # Is this step supported by previous evidence?

@dataclass
class TrajectoryEval:
    run_id: str
    steps: list[TrajectoryStep]
    outcome: Literal["success", "failure"]

    def score(self) -> dict:
        tool_calls = [s for s in self.steps if s.action == "tool_call"]
        grounded   = [s for s in self.steps if s.grounded]
        
        return {
            "outcome":          self.outcome,
            "tool_efficiency":  len(tool_calls) / max(len(self.steps), 1),
            "grounding_rate":   len(grounded)   / max(len(self.steps), 1),
            "unnecessary_calls": sum(1 for s in tool_calls if not s.grounded),
            "trajectory_score": (
                (1.0 if self.outcome == "success" else 0.0) * 0.40 +
                (len(grounded) / max(len(self.steps), 1)) * 0.40 +
                (1 - len(tool_calls)/max(len(self.steps),1)) * 0.20
            ),
        }

# Good trajectory: concise, grounded, correct outcome
good = TrajectoryEval("run-A", [
    TrajectoryStep(1, "think",     "Check error rate first",                  True),
    TrajectoryStep(2, "tool_call", "query_metrics(checkout-ui)",              True),
    TrajectoryStep(3, "think",     "Error rate 31% - check deployment",       True),
    TrajectoryStep(4, "tool_call", "get_deployment(checkout-ui)",             True),
    TrajectoryStep(5, "output",    "Hypothesis: v2.1 broke 3DS. PROPOSAL...", True),
], outcome="success")

# Bad trajectory: unnecessary calls, hallucinated reasoning, lucky outcome
bad = TrajectoryEval("run-B", [
    TrajectoryStep(1, "think",     "Let me check everything just to be safe",  True),
    TrajectoryStep(2, "tool_call", "query_metrics(checkout-ui)",               True),
    TrajectoryStep(3, "tool_call", "query_metrics(billing-api)",               False),  # unnecessary
    TrajectoryStep(4, "tool_call", "query_metrics(auth-service)",              False),  # unnecessary
    TrajectoryStep(5, "think",     "The root cause is Redis (not supported by any evidence)", False),  # hallucination
    TrajectoryStep(6, "tool_call", "get_deployment(checkout-ui)",              True),
    TrajectoryStep(7, "output",    "Hypothesis: v2.1 broke 3DS. PROPOSAL...", True),
], outcome="success")   # got lucky — same correct answer despite bad reasoning

print("📐  Trajectory Analysis Demo")
print("=" * 60)

for traj in [good, bad]:
    scores = traj.score()
    print(f"\n  Run: {traj.run_id}  Outcome: {traj.outcome}")
    for k, v in scores.items():
        if isinstance(v, float):
            print(f"    {k:<25} {v:.2f}")
        else:
            print(f"    {k:<25} {v}")

print()
print("  KEY INSIGHT: Both runs have 'success' outcome, but run-B scores")
print("  significantly lower on trajectory quality. In production, run-B's")
print("  approach will eventually produce incorrect outcomes for edge cases.")


📐  Trajectory Analysis Demo

  Run: run-A  Outcome: success
    outcome                   success
    tool_efficiency           0.40
    grounding_rate            1.00
    unnecessary_calls         0
    trajectory_score          0.92

  Run: run-B  Outcome: success
    outcome                   success
    tool_efficiency           0.57
    grounding_rate            0.57
    unnecessary_calls         2
    trajectory_score          0.71

  KEY INSIGHT: Both runs have 'success' outcome, but run-B scores
  significantly lower on trajectory quality. In production, run-B's
  approach will eventually produce incorrect outcomes for edge cases.
